<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/2024_08_21_XGBoost_on_ImageJ_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install xgboost

import json
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, AutoTokenizer, get_scheduler
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from tqdm.notebook import tqdm, trange
from time import perf_counter
from PIL import Image
import pandas as pd
#from google.colab import drive
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle

print("All dependencies present.")

   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   - -------------------------------------- 2.1/72.0 MB 13.2 MB/s eta 0:00:06
   ----- ---------------------------------- 9.4/72.0 MB 23.7 MB/s eta 0:00:03
   --------- ------------------------------ 16.3/72.0 MB 21.8 MB/s eta 0:00:03
   --------------- ------------------------ 27.5/72.0 MB 27.5 MB/s eta 0:00:02
   ------------------ --------------------- 32.8/72.0 MB 25.7 MB/s eta 0:00:02
   --------------------- ------------------ 38.5/72.0 MB 25.8 MB/s eta 0:00:02
   ------------------------ --------------- 44.8/72.0 MB 25.7 MB/s eta 0:00:02
   ------------------------- -------------- 46.7/72.0 MB 22.3 MB/s eta 0:00:02
   -------------------------- ------------- 47.4/72.0 MB 19.7 MB/s eta 0:00:02
   -------------------------- ------------- 48.2/72.0 MB 18.1 MB/s eta 0:00:02
   --------------------------- ------------ 50.1/72.0 MB 16.7 MB/s eta 0:00:02
   ------------------------------ --------- 55.6/72.0 MB 15.9 M

ModuleNotFoundError: No module named 'torch'

In [ ]:
# set random seeds for repeatability
import numpy as np
import random

def set_seed(seed_val):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)

print("Seed set")

Seed set


In [ ]:
seed_val = 42
set_seed(seed_val)

In [ ]:
file_in = "data/Images_Classified_np126"

with open(file_in + '.pkl', 'rb') as f:
    feats_df = pickle.load(f)

#load in dataframe
pd.set_option('display.max_columns', None)

file_out = file_in + "_s" + str(seed_val) + "_XGB" + ".model"

print(file_out)
print(feats_df.shape)

feats_df.head()


data/Images_Classified_np126_s42_XGB.model
(27515, 113)


,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,seed,dir,feat_bool,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,Y_mean,Y_std,Perim._mean,Perim._std,BX_mean,BX_std,BY_mean,BY_std,Width_mean,Width_std,Height_mean,Height_std,Major_mean,Major_std,Minor_mean,Minor_std,Angle_mean,Angle_std,Circ._mean,Circ._std,Feret_mean,Feret_std,IntDen_mean,IntDen_std,%Area_mean,%Area_std,RawIntDen_mean,RawIntDen_std,FeretX_mean,FeretX_std,FeretY_mean,FeretY_std,FeretAngle_mean,FeretAngle_std,MinFeret_mean,MinFeret_std,AR_mean,AR_std,Round_mean,Round_std,Solidity_mean,Solidity_std,num_spots_inverted,Mean_inverted,Median_inverted,Area_inverted_mean,Area_inverted_std,X_inverted_mean,X_inverted_std,Y_inverted_mean,Y_inverted_std,Perim._inverted_mean,Perim._inverted_std,BX_inverted_mean,BX_inverted_std,BY_inverted_mean,BY_inverted_std,Width_inverted_mean,Width_inverted_std,Height_inverted_mean,Height_inverted_std,Major_inverted_mean,Major_inverted_std,Minor_inverted_mean,Minor_inverted_std,Angle_inverted_mean,Angle_inverted_std,Circ._inverted_mean,Circ._inverted_std,Feret_inverted_mean,Feret_inverted_std,IntDen_inverted_mean,IntDen_inverted_std,%Area_inverted_mean,%Area_inverted_std,RawIntDen_inverted_mean,RawIntDen_inverted_std,FeretX_inverted_mean,FeretX_inverted_std,FeretY_inverted_mean,FeretY_inverted_std,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std,full_path,classifier_pred_class
0,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,1,/content/Images3,1,70,5.489,0,12.300000,2.548669,99.395943,58.506181,97.530229,58.913237,12.282143,1.400962,97.371429,58.453442,95.528571,58.843551,4.071429,0.723512,4.057143,0.753766,4.245571,0.439294,3.664200,0.511781,55.108900,57.413924,0.958686,0.069883,4.926700,0.412294,3136.500000,649.910626,100.0,0.0,3136.500000,649.910626,97.942857,58.572760,96.014286,58.843981,120.129000,30.775307,3.653171,0.564778,1.177686,0.183575,0.866686,0.115368,0.879486,0.067138,1,249.511,255,39139.0,0.0,99.998,0.0,100.022,0.0,811.882,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.351,0.0,223.117,0.0,38.796,0.0,0.746,0.0,282.843,0.0,9980445.0,0.0,100.0,0.0,9980445.0,0.0,0.0,0.0,200.0,0.0,45.0,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0,/content/Images3/1.png,2
1,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,3,/content/Images3,1,63,4.921,0,12.253968,2.569468,101.800571,59.297600,99.451270,58.953924,11.663270,1.365051,99.968254,59.323693,97.619048,58.964458,3.666667,0.534522,3.666667,0.534522,4.237317,0.401713,3.661714,0.572434,56.049556,60.605082,0.999619,0.003000,4.908365,0.491745,3124.761905,655.214389,100.0,0.0,3124.761905,655.214389,100.158730,59.330446,98.079365,59.017431,120.191190,32.406426,3.476190,0.613529,1.185032,0.226780,0.867667,0.127976,0.954905,0.046408,1,250.078,255,39228.0,0.0,99.992,0.0,100.015,0.0,809.255,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.524,0.0,223.451,0.0,72.380,0.0,0.753,0.0,282.843,0.0,10003140.0,0.0,100.0,0.0,10003140.0,0.0,0.0,0.0,0.0,0.0,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.981,0.0,/content/Images3/3.png,2
2,0.028367,0.057912,0.073998,0.080784,-0.133789,0.008512,0.534244,1,2,7.png,7,/content/Images3,1,77,4.692,0,9.558442,1.427863,100.557675,58.022388,98.103338,59.633687,10.379961,1.091819,98.870130,58.006796,96.519481,59.576560,3.402597,0.564300,3.194805,0.559196,3.806623,0.403173,3.196818,0.352550,29.220831,46.845042,0.992974,0.032896,4.416805,0.320359,2437.402597,364.105032,100.0,0.0,2437.402597,364.105032,98.961039,57.996292,96.753247,59.526521,133.450130,23.616781,2.961039,0.375951,1.209857,0.220792,0.849429,0.129524,0.946805,0.050966,1,250.308,255,39264.0,0.0,99.987,0.0,100.001,0.0,808.326,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.626,0.0,223.554,0.0,5.766,0.0,0.755,0.0,282.843,0.0,10012320.0,0.0,100.0,0.0,10012320.0,0.0,0.0,0.0,200.0,0.0,45.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,

In [35]:
print(feats_df.shape)
feats_df.head()

(27515, 113)


,Ua,Ui,Ga,Gi,Ba,Da,Di,pattern,noise,path,seed,dir,feat_bool,num_spots,Mean,Median,Area_mean,Area_std,X_mean,X_std,Y_mean,Y_std,Perim._mean,Perim._std,BX_mean,BX_std,BY_mean,BY_std,Width_mean,Width_std,Height_mean,Height_std,Major_mean,Major_std,Minor_mean,Minor_std,Angle_mean,Angle_std,Circ._mean,Circ._std,Feret_mean,Feret_std,IntDen_mean,IntDen_std,%Area_mean,%Area_std,RawIntDen_mean,RawIntDen_std,FeretX_mean,FeretX_std,FeretY_mean,FeretY_std,FeretAngle_mean,FeretAngle_std,MinFeret_mean,MinFeret_std,AR_mean,AR_std,Round_mean,Round_std,Solidity_mean,Solidity_std,num_spots_inverted,Mean_inverted,Median_inverted,Area_inverted_mean,Area_inverted_std,X_inverted_mean,X_inverted_std,Y_inverted_mean,Y_inverted_std,Perim._inverted_mean,Perim._inverted_std,BX_inverted_mean,BX_inverted_std,BY_inverted_mean,BY_inverted_std,Width_inverted_mean,Width_inverted_std,Height_inverted_mean,Height_inverted_std,Major_inverted_mean,Major_inverted_std,Minor_inverted_mean,Minor_inverted_std,Angle_inverted_mean,Angle_inverted_std,Circ._inverted_mean,Circ._inverted_std,Feret_inverted_mean,Feret_inverted_std,IntDen_inverted_mean,IntDen_inverted_std,%Area_inverted_mean,%Area_inverted_std,RawIntDen_inverted_mean,RawIntDen_inverted_std,FeretX_inverted_mean,FeretX_inverted_std,FeretY_inverted_mean,FeretY_inverted_std,FeretAngle_inverted_mean,FeretAngle_inverted_std,MinFeret_inverted_mean,MinFeret_inverted_std,AR_inverted_mean,AR_inverted_std,Round_inverted_mean,Round_inverted_std,Solidity_inverted_mean,Solidity_inverted_std,full_path,classifier_pred_class
0,0.025122,0.063822,0.071957,0.106327,-0.113131,0.009186,0.611871,1,2,1.png,1,/content/Images3,1,70,5.489,0,12.300000,2.548669,99.395943,58.506181,97.530229,58.913237,12.282143,1.400962,97.371429,58.453442,95.528571,58.843551,4.071429,0.723512,4.057143,0.753766,4.245571,0.439294,3.664200,0.511781,55.108900,57.413924,0.958686,0.069883,4.926700,0.412294,3136.500000,649.910626,100.0,0.0,3136.500000,649.910626,97.942857,58.572760,96.014286,58.843981,120.129000,30.775307,3.653171,0.564778,1.177686,0.183575,0.866686,0.115368,0.879486,0.067138,1,249.511,255,39139.0,0.0,99.998,0.0,100.022,0.0,811.882,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.351,0.0,223.117,0.0,38.796,0.0,0.746,0.0,282.843,0.0,9980445.0,0.0,100.0,0.0,9980445.0,0.0,0.0,0.0,200.0,0.0,45.0,0.0,200.0,0.0,1.001,0.0,0.999,0.0,0.979,0.0,/content/Images3/1.png,2
1,0.034475,0.059956,0.080798,0.10037,-0.132577,0.008852,0.854084,1,2,3.png,3,/content/Images3,1,63,4.921,0,12.253968,2.569468,101.800571,59.297600,99.451270,58.953924,11.663270,1.365051,99.968254,59.323693,97.619048,58.964458,3.666667,0.534522,3.666667,0.534522,4.237317,0.401713,3.661714,0.572434,56.049556,60.605082,0.999619,0.003000,4.908365,0.491745,3124.761905,655.214389,100.0,0.0,3124.761905,655.214389,100.158730,59.330446,98.079365,59.017431,120.191190,32.406426,3.476190,0.613529,1.185032,0.226780,0.867667,0.127976,0.954905,0.046408,1,250.078,255,39228.0,0.0,99.992,0.0,100.015,0.0,809.255,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.524,0.0,223.451,0.0,72.380,0.0,0.753,0.0,282.843,0.0,10003140.0,0.0,100.0,0.0,10003140.0,0.0,0.0,0.0,0.0,0.0,135.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,0.981,0.0,/content/Images3/3.png,2
2,0.028367,0.057912,0.073998,0.080784,-0.133789,0.008512,0.534244,1,2,7.png,7,/content/Images3,1,77,4.692,0,9.558442,1.427863,100.557675,58.022388,98.103338,59.633687,10.379961,1.091819,98.870130,58.006796,96.519481,59.576560,3.402597,0.564300,3.194805,0.559196,3.806623,0.403173,3.196818,0.352550,29.220831,46.845042,0.992974,0.032896,4.416805,0.320359,2437.402597,364.105032,100.0,0.0,2437.402597,364.105032,98.961039,57.996292,96.753247,59.526521,133.450130,23.616781,2.961039,0.375951,1.209857,0.220792,0.849429,0.129524,0.946805,0.050966,1,250.308,255,39264.0,0.0,99.987,0.0,100.001,0.0,808.326,0.0,0.0,0.0,0.0,0.0,200.0,0.0,200.0,0.0,223.626,0.0,223.554,0.0,5.766,0.0,0.755,0.0,282.843,0.0,10012320.0,0.0,100.0,0.0,10012320.0,0.0,0.0,0.0,200.0,0.0,45.0,0.0,200.0,0.0,1.000,0.0,1.000,0.0,

In [36]:
print(feats_df.columns[13:111])

Index(['num_spots', 'Mean', 'Median', 'Area_mean', 'Area_std', 'X_mean',
       'X_std', 'Y_mean', 'Y_std', 'Perim._mean', 'Perim._std', 'BX_mean',
       'BX_std', 'BY_mean', 'BY_std', 'Width_mean', 'Width_std', 'Height_mean',
       'Height_std', 'Major_mean', 'Major_std', 'Minor_mean', 'Minor_std',
       'Angle_mean', 'Angle_std', 'Circ._mean', 'Circ._std', 'Feret_mean',
       'Feret_std', 'IntDen_mean', 'IntDen_std', '%Area_mean', '%Area_std',
       'RawIntDen_mean', 'RawIntDen_std', 'FeretX_mean', 'FeretX_std',
       'FeretY_mean', 'FeretY_std', 'FeretAngle_mean', 'FeretAngle_std',
       'MinFeret_mean', 'MinFeret_std', 'AR_mean', 'AR_std', 'Round_mean',
       'Round_std', 'Solidity_mean', 'Solidity_std', 'num_spots_inverted',
       'Mean_inverted', 'Median_inverted', 'Area_inverted_mean',
       'Area_inverted_std', 'X_inverted_mean', 'X_inverted_std',
       'Y_inverted_mean', 'Y_inverted_std', 'Perim._inverted_mean',
       'Perim._inverted_std', 'BX_inverted_mean', 'BX_

XGBOOST

In [37]:
#! pip install xgboost
#! pip install openpyxl
#! pip install tensorflow
#! pip install graphviz
#! pip install hyperopt

from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
import numpy as np
from sklearn.model_selection import RepeatedKFold
from numpy import absolute
from pandas import read_csv
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from xgboost import XGBRegressor
import openpyxl
from xgboost import cv
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import operator
# for loading/processing the images
import tensorflow
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from keras.applications.vgg16 import preprocess_input

# models
from keras.applications.vgg16 import VGG16
from keras.models import Model

# clustering and dimension reduction
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pickle
from sklearn.ensemble import RandomForestRegressor
from sklearn import tree
import graphviz
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from hyperopt import fmin, tpe, hp,STATUS_OK
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

print("All dependencies present")

All dependencies present


In [39]:
# scale data 
# uses Standard Scaler, range [-1, 1]

## X (independent variable) --> the feature values themselves
X = feats_df[feats_df.columns[13:111]].astype(float)

## Y (dependent variables) --> the parameter values of Negan and RTO themselves
y = feats_df[["Ua","Ui","Ga","Gi","Da","Di","Ba"]].astype(float)

#Scale data with standardscaler
scaling = StandardScaler()

# Use fit and transform method
scaling.fit(X)
X_scaled = scaling.transform(X)

# select 20 percent for testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42) #create training split


In [40]:
print(X_test)
#print(y_train)

[[-0.33573772 -0.89290703 -0.79679244 ... -0.27982889  0.28731207
  -0.24394859]
 [-0.56983029  0.9038021   1.25503199 ...  4.04241354 -0.40972916
   3.12418153]
 [-0.63225498 -0.9682853  -0.79679244 ... -0.27982889  0.5618212
  -0.24394859]
 ...
 [-0.61664881  1.31238128  1.25503199 ... -0.27982889  0.5618212
  -0.24394859]
 [-0.63225498 -0.9682853  -0.79679244 ... -0.27982889  0.5618212
  -0.24394859]
 [-0.61664881  1.31238128  1.25503199 ... -0.27982889  0.5618212
  -0.24394859]]


In [41]:
y_test

,Ua,Ui,Ga,Gi,Da,Di,Ba
6706,0.046849,0.013064,0.115779,0.033348,0.011170,0.616537,-0.118356
19387,0.022846,0.156399,0.118046,0.069050,0.009939,1.052442,-0.193037
14847,0.045896,0.199486,0.061461,0.078881,0.009406,0.433193,-0.097045
24306,0.024930,0.051862,0.094625,0.024345,0.010236,0.498789,-0.098016
7172,0.030000,0.070000,0.080000,0.100000,0.010000,0.827444,-0.120000
...,...,...,...,...,...,...,...
12968,0.014529,0.138111,0.112518,0.063431,0.010815,1.113066,-0.100108
1891,0.009662,0.195854,0.053540,0.101883,0.011343,1.117866,-0.109434
23541,0.045216,0.168554,0.076138,0.056090,0.011093,0.641815,-0.057450
16741,0.020215,0.142634,0.059699,0.142708,0.010567,0.614994,-0.108160


In [42]:
model = xgb.XGBRegressor(n_estimators=1000, max_depth=10, eta=0.1, subsample=0.7, colsample_bytree=0.8, num_boost_round=50, objective= "reg:squarederror", device = "cuda")
model.fit(X_train, y_train)

/Users/mikhailblinov/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [14:08:54] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)
/Users/mikhailblinov/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [14:08:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "num_boost_round" } are not used.

  warnings.warn(smsg, UserWarning)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eta=0.1, eval_metric=None,
             feature_types=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=None, num_boost_round=50, ...)

In [43]:
## Save XGBoost Model to File
model.save_model(file_out)

/Users/mikhailblinov/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [14:09:23] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


In [44]:
loaded_model = xgb.XGBRegressor()

loaded_model.load_model(file_out)

#loaded_model.fit(X_train, y_train)

# make predictions
y_pred = loaded_model.predict(X_test)

pd.DataFrame(data=y_pred) 

# Columns - parameters 1-7
# Rows - Subsample of rows

print("Model loaded")

Model loaded


In [45]:
# GOAL: To determine the error rate in predicting the value of continuous, parameter data

# Converts test Y values to numpy array
true_vals = np.array(y_test)

# Outputs amount of true values
print(true_vals.shape)

# Outputs amount of predicted values
print(y_pred.shape)

# Sets counters for correct, incorrect, errors numpy array
correct = 0
incorrect = 0
p_errs = np.zeros(7)

# loops through each row in the true_vals array
for i in range(true_vals.shape[0]):
  # predicted value = predicted value from loop
  pred = y_pred[i]
  # true value = true value from loop
  true_val = true_vals[i]
  # Adds to error: absolute percent difference between true and predicted values
  p_errs += (np.abs((pred-true_val)/true_val))

# Outputs error percentages
print((p_errs/true_vals.shape[0])*100)

(5503, 7)
(5503, 7)
[32.87151772 58.13106727 26.3401836  41.73659681  8.92654801 38.65788645
 33.91673308]


In [ ]:
# select features of real images

#new_df = pd.read_pickle("data/real_df_xgboost2.pkl")
#new_feats = new_df[new_df.columns[3:111]]

new_df = pd.read_pickle("./data/Images_Classified_np126.pkl")
new_feats = new_df[new_df.columns[13:111]]

print(new_feats.head())

y_pred = None

loaded_model = xgb.XGBRegressor()

loaded_model.load_model(file_out)


#scale data
scaling=StandardScaler()

# Use fit and transform method
scaling.fit(new_feats)
new_feats_scaled = scaling.transform(new_feats)
print(new_feats_scaled.shape)

# predict params of images from real feats
y_pred = loaded_model.predict(new_feats_scaled)
print(y_pred[2])

new_df["pred_params"] = list(y_pred)

   num_spots   Mean  Median  Area_mean  Area_std      X_mean      X_std  \
0         70  5.489       0  12.300000  2.548669   99.395943  58.506181   
1         63  4.921       0  12.253968  2.569468  101.800571  59.297600   
2         77  4.692       0   9.558442  1.427863  100.557675  58.022388   
3         73  7.931       0  17.041096  3.365572  103.192973  59.580318   
4         75  8.154       0  17.053333  3.905187  104.204653  61.052503   

      Y_mean      Y_std  Perim._mean  Perim._std     BX_mean     BX_std  \
0  97.530229  58.913237    12.282143    1.400962   97.371429  58.453442   
1  99.451270  58.953924    11.663270    1.365051   99.968254  59.323693   
2  98.103338  59.633687    10.379961    1.091819   98.870130  58.006796   
3  99.320973  58.524761    13.973644    1.464372  100.917808  59.675321   
4  99.000587  59.296666    14.229160    1.984575  102.000000  61.070014   

     BY_mean     BY_std  Width_mean  Width_std  Height_mean  Height_std  \
0  95.528571  58.843551

In [ ]:
new_df

In [64]:
## TODO: Grab dataframe at the top (w/ all the features), drop all unnessesary columns (diff. from real_df, drop parameters)
## choose first five images and run it and see how close they are. 

print ("Ua","Ui","Ga","Gi","Da","Di","Ba")
for i in range(new_df.shape[0]):
  print(new_df["path"][i])
  print(new_df["pred_params"][i])
  print(new_df["Ua"][i], new_df["Ui"][i],new_df["Ga"][i], new_df["Gi"][i],new_df["Da"][i], new_df["Di"][i],new_df["Ba"][i]
      ) 


Ua Ui Ga Gi Da Di Ba
1.png
[ 0.02490418  0.0640335   0.07262938  0.10606965  0.0097551   0.61184
 -0.11362164]
0.02512223660069473 0.06382166631976507 0.07195706357833266 0.10632739510186466 0.009185636723970254 0.6118714917866803 -0.11313134072543651
3.png
[ 0.03404492  0.06016414  0.08108194  0.1010063   0.00927781  0.8539903
 -0.1320489 ]
0.03447465809161778 0.0599556426453826 0.0807983471177822 0.10037032303039066 0.008852352175844071 0.8540837792143913 -0.13257673761191435
7.png
[ 0.02888181  0.05787217  0.07419606  0.08122779  0.00906289  0.5339657
 -0.13312519]
0.028366614907530105 0.057912397008865274 0.07399764894945274 0.08078401006827379 0.008512024948034682 0.5342443632295809 -0.1337886113318717
12.png
[ 0.03620365  0.06268869  0.07712788  0.09331793  0.01043348  0.90076303
 -0.11233992]
0.035846536188983404 0.07876223021258162 0.07822983359535865 0.10148492077956844 0.010542271530694876 0.9338941354749168 -0.11575188943822867
13.png
[ 0.02987736  0.06755836  0.07887999  0.